# NoveltyWekan Integrated Optimization Pipeline

This notebook is an end-to-end medical image segmentation pipeline. It integrates the baseline Stanford ML **Auto-Generate-WLs** framework with **8 advanced optimizations** to handle extreme label scarcity on the Breast Ultrasound Images (BUSI) dataset:

1. **SimCLR-style Contrastive Pretraining** of the UNet++ encoder on unlabeled images.
2. **Seed Locking & Reproducibility:** Multi-seed training synchronization.
3. **Complexity Gating:** Classifier predicting MedSAM segmentation success.
4. **Augmentation Consistency Check:** Test-Time-Augmentation (TTA) using rotated coarse predictions.
5. **Centroid-guided MedSAM Prompts:** Bounding Box + Centroid point prompts.
6. **V2 Morphological Gates:** Geometric filtering (stripe, aspect, convexity, size, and components).
7. **Radiomics Quality Predictor:** Grading weak labels using shape/texture regression.
8. **Curriculum-Ordered Generation:** Easy-to-hard progressive generation in batches.
9. **Geometry-Aware Boundary Loss Weighting:** Distance-weighted BCE loss.


## Section 1: Mount Drive & Path Setup


In [ ]:
# =================================================================
# SECTION 1: Mount Drive & Setup — FULL BUSI DATASET VERSION
# =================================================================
from google.colab import drive
import os

drive.mount('/content/drive')

# Check possible paths in Drive for both names (Database vs Dataset)
drive_paths = [
    "/content/drive/MyDrive/inputs/Database_BUSI_with_GT",
    "/content/drive/MyDrive/inputs/Dataset_BUSI_with_GT"
]

BUSI_ROOT = None
for p in drive_paths:
    if os.path.exists(p):
        BUSI_ROOT = p
        break

assert BUSI_ROOT is not None, "Could not find Database_BUSI_with_GT or Dataset_BUSI_with_GT in Google Drive inputs folder! Please verify your path."

# Auto-detect nesting (e.g., benign/benign vs benign)
benign_dir = os.path.join(BUSI_ROOT, "benign")
if os.path.isdir(os.path.join(benign_dir, "benign")):
    benign_dir = os.path.join(benign_dir, "benign")

malignant_dir = os.path.join(BUSI_ROOT, "malignant")
if os.path.isdir(os.path.join(malignant_dir, "malignant")):
    malignant_dir = os.path.join(malignant_dir, "malignant")

# Count only the actual images (not masks)
benign_imgs    = sorted([f for f in os.listdir(benign_dir)    if f.endswith('.png') and '_mask' not in f])
malignant_imgs = sorted([f for f in os.listdir(malignant_dir) if f.endswith('.png') and '_mask' not in f])

print(f"Dataset Root Found: {BUSI_ROOT}")
print(f"Benign images:      {len(benign_imgs)}")
print(f"Malignant images:   {len(malignant_imgs)}")
print(f"Total (B+M):        {len(benign_imgs) + len(malignant_imgs)}")
print("✅ Dataset verified. Ready for splitting.")

Mounted at /content/drive
Dataset Root Found: /content/drive/MyDrive/inputs/Dataset_BUSI_with_GT
Benign images:      437
Malignant images:   210
Total (B+M):        647
✅ Dataset verified. Ready for splitting.


## Section 2: Clone Repositories & Install Dependencies


In [ ]:
# Clone Stanford ML Weak Label Repo
!rm -rf Auto-Generate-WLs
!git clone https://github.com/stanfordmlgroup/Auto-Generate-WLs
# Clone Bowang Lab MedSAM Repo
!rm -rf MedSAM
!git clone https://github.com/bowang-lab/MedSAM

# Install Dependencies
!pip install segment-anything opencv-python matplotlib scikit-image scikit-learn
!pip install albumentations==0.5.2
!pip install imgaug==0.4.0
!pip install -e MedSAM/


Cloning into 'Auto-Generate-WLs'...
remote: Enumerating objects: 1836, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 1836 (delta 12), reused 13 (delta 4), pack-reused 1809 (from 1)
Receiving objects: 100% (1836/1836), 908.05 MiB | 18.10 MiB/s, done.
Resolving deltas: 100% (127/127), done.
Updating files: 100% (3275/3275), done.
Cloning into 'MedSAM'...
remote: Enumerating objects: 967, done.
remote: Total 967 (delta 0), reused 0 (delta 0), pack-reused 967 (from 1)
Receiving objects: 100% (967/967), 62.91 MiB | 17.73 MiB/s, done.
Resolving deltas: 100% (475/475), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 65.6 MB/s eta 0:00:00
  Attempting uninstall: albumentations
    Found existing installation: albumentations 2.0.8
    Uninstalling albumentations-2.0.8:
      Successfully uninstalled albumentations-2.0.8
Obtaining fil

## Section 3: Patch imgaug numpy compatibility


In [ ]:
# Solve numpy.sctypes deprecation in imgaug (critical patch)
!sed -i 's/np.sctypes\["float"\]/[np.float32, np.float64]/g' /usr/local/lib/python3.12/dist-packages/imgaug/imgaug.py
!sed -i 's/np.sctypes\["int"\]/[np.int32, np.int64]/g' /usr/local/lib/python3.12/dist-packages/imgaug/imgaug.py
!sed -i 's/np.sctypes\["uint"\]/[np.uint8, np.uint16]/g' /usr/local/lib/python3.12/dist-packages/imgaug/imgaug.py
print("imgaug patched successfully.")


imgaug patched successfully.


## Section 4: Split Gold Standard into Train (20) and Test (5) (Leakage Prevention)


In [ ]:
# =================================================================
# SECTION 4: Full BUSI Dataset Split — Leakage-Free (Paper Setup)
# =================================================================
import os, shutil, random
import cv2
import numpy as np

random.seed(42)
np.random.seed(42)

# Collect all images (no masks) from the directories resolved in Section 1
all_imgs = [('benign', f) for f in benign_imgs] + [('malignant', f) for f in malignant_imgs]
# Sort to ensure deterministic shuffling across runs/environments
all_imgs.sort(key=lambda x: (x[0], x[1]))
random.shuffle(all_imgs)

total = len(all_imgs)
n_test = int(total * 0.20)   # ~130 test images (paper uses 80-20 split)
n_gs   = 25                   # Gold Standard (matches paper)

test_set  = all_imgs[:n_test]
train_set = all_imgs[n_test:]
gs_set    = train_set[:n_gs]          # First 25 of training = Gold Standard
unlabeled = train_set[n_gs:]          # Remaining = Unlabeled pool for WL generation

print(f"Total images:        {total}")
print(f"Test set (locked):   {len(test_set)}")
print(f"GS training set:     {len(gs_set)}")
print(f"Unlabeled pool:      {len(unlabeled)}")

# Merge multiple masks using logical OR and save
def merge_and_save_mask(img_name, src_folder, dest_path):
    base_name = os.path.splitext(img_name)[0]
    mask_files = [
        os.path.join(src_folder, f)
        for f in os.listdir(src_folder)
        if f.startswith(base_name + "_mask") and f.endswith(".png")
    ]
    if not mask_files:
        raise FileNotFoundError(f"No mask files found for {img_name} in {src_folder}")

    # Merge using bitwise OR
    merged = cv2.imread(mask_files[0], cv2.IMREAD_GRAYSCALE)
    for f in mask_files[1:]:
        m = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        merged = cv2.bitwise_or(merged, m)

    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    cv2.imwrite(dest_path, merged)

base = "Auto-Generate-WLs/pytorch-nested-unet/inputs"

def copy_split(split_data, split_name):
    img_out  = f"{base}/{split_name}/images"
    mask_out = f"{base}/{split_name}/masks/0"
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(mask_out, exist_ok=True)

    for cls, fname in split_data:
        src_dir  = benign_dir if cls == 'benign' else malignant_dir
        img_src  = os.path.join(src_dir, fname)

        # Clean name to match model expectation (double underscore format)
        clean_name = fname.replace(' (', '__').replace(')', '_') \
                          .replace('(', '').replace(' ', '')
        out_name = f"{cls}__{clean_name}" if not clean_name.startswith(cls) else clean_name

        # Copy image
        shutil.copy(img_src,  f"{img_out}/{out_name}")

        # Merge and save mask
        merge_and_save_mask(fname, src_dir, f"{mask_out}/{out_name}")
    print(f"  Created '{split_name}': {len(split_data)} images")

copy_split(test_set,  "busi-test-130")     # 🔒 Locked test set — never train on this
copy_split(gs_set,    "busi-gs-25")        # Gold Standard training set
copy_split(unlabeled, "busi-unlabeled")    # Unlabeled pool for weak label generation

# Verify zero leakage
test_names     = set(os.listdir(f"{base}/busi-test-130/images"))
gs_names       = set(os.listdir(f"{base}/busi-gs-25/images"))
unlabeled_names = set(os.listdir(f"{base}/busi-unlabeled/images"))

print(f"\n=== LEAKAGE CHECK ===")
print(f"Test ∩ GS:         {len(test_names & gs_names)}  (must be 0)")
print(f"Test ∩ Unlabeled:  {len(test_names & unlabeled_names)}  (must be 0)")
print(f"GS ∩ Unlabeled:    {len(gs_names & unlabeled_names)}  (must be 0)")
if not (test_names & gs_names) and not (test_names & unlabeled_names):
    print("✅ ZERO LEAKAGE CONFIRMED. Safe to proceed.")
else:
    print("🚨 LEAKAGE DETECTED! Check your split logic.")

Total images:        647
Test set (locked):   129
GS training set:     25
Unlabeled pool:      493
  Created 'busi-test-130': 129 images
  Created 'busi-gs-25': 25 images
  Created 'busi-unlabeled': 493 images

=== LEAKAGE CHECK ===
Test ∩ GS:         0  (must be 0)
Test ∩ Unlabeled:  0  (must be 0)
GS ∩ Unlabeled:    0  (must be 0)
✅ ZERO LEAKAGE CONFIRMED. Safe to proceed.


## Section 5: Contrastive Pretraining of U-Net++ Encoder (Idea 8)
Pretrains U-Net++ encoder using NT-Xent loss on the unlabeled images prior to segmentation training.


In [ ]:
# Write pretrain_contrastive.py to pytorch-nested-unet/
contrastive_pretrain_code = """
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import cv2
import os

class UnlabeledDataset(Dataset):
    def __init__(self, img_dir, img_ext='.png'):
        self.img_dir = img_dir
        self.img_ids = [f for f in os.listdir(img_dir) if f.endswith(img_ext)]

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_ids[idx])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (256, 256))
        img = img.astype('float32') / 255.0
        img = img.transpose(2, 0, 1) # to (C, H, W)
        return {'image': torch.tensor(img)}

class ContrastiveUNetPretrainer(nn.Module):
    def __init__(self, unet_model):
        super().__init__()
        self.unet = unet_model
        # Projector head maps bottleneck (512 features) to projection space
        self.projector = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )

    def forward(self, x):
        # NestedUNet encoder layers
        x0_0 = self.unet.conv0_0(x)
        x1_0 = self.unet.conv1_0(self.unet.pool(x0_0))
        x2_0 = self.unet.conv2_0(self.unet.pool(x1_0))
        x3_0 = self.unet.conv3_0(self.unet.pool(x2_0))
        x4_0 = self.unet.conv4_0(self.unet.pool(x3_0))
        z = self.projector(x4_0)
        return nn.functional.normalize(z, dim=1)

def nt_xent_loss(z1, z2, temperature=0.5):
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.t()) / temperature
    n = z1.shape[0]
    labels = torch.cat([
        torch.arange(n, 2 * n),
        torch.arange(0, n)
    ]).to(z.device)
    mask = torch.eye(2 * n, dtype=torch.bool).to(z.device)
    sim.masked_fill_(mask, -9e15)
    return nn.CrossEntropyLoss()(sim, labels)

def pretrain(unet_model, unlabeled_dir, device, epochs=15, lr=1e-3):
    dataset = UnlabeledDataset(unlabeled_dir)
    loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, drop_last=True)
    model = ContrastiveUNetPretrainer(unet_model).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    aug = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomRotation(30),
    ])

    print("Beginning Contrastive pretraining on unlabeled set...")
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch in loader:
            imgs = batch['image'].to(device)
            view1 = aug(imgs)
            view2 = aug(imgs)
            z1 = model(view1)
            z2 = model(view2)
            loss = nt_xent_loss(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"Pretrain Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(loader):.4f}")
    return model.unet
"""
with open("Auto-Generate-WLs/pytorch-nested-unet/pretrain_contrastive.py", "w") as f:
    f.write(contrastive_pretrain_code)
print("Contrastive pretraining module written to disk.")


Contrastive pretraining module written to disk.


In [ ]:
# Run pretraining setup on UNet++ architecture
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
import torch
import archs
from pretrain_contrastive import pretrain

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
unet_model = archs.NestedUNet(num_classes=1, input_channels=3, deep_supervision=False)

# Use our new unlabeled split (which has ~492 images instead of 25)
unlabeled_images_dir = "inputs/busi-unlabeled/images"

# Run 15 epochs of SimCLR pretraining
pretrained_unet = pretrain(unet_model, unlabeled_images_dir, device, epochs=15)
torch.save(pretrained_unet.state_dict(), "models/pretrained_unet_encoder.pth")
print("Pretrained encoder weights saved.")
%cd /content

/content/Auto-Generate-WLs/pytorch-nested-unet
Beginning Contrastive pretraining on unlabeled set...
Pretrain Epoch 1/15 | Loss: 0.7060
Pretrain Epoch 2/15 | Loss: 0.6390
Pretrain Epoch 3/15 | Loss: 0.6077
Pretrain Epoch 4/15 | Loss: 0.5981
Pretrain Epoch 5/15 | Loss: 0.5730
Pretrain Epoch 6/15 | Loss: 0.5512
Pretrain Epoch 7/15 | Loss: 0.5257
Pretrain Epoch 8/15 | Loss: 0.5249
Pretrain Epoch 9/15 | Loss: 0.5178
Pretrain Epoch 10/15 | Loss: 0.5463
Pretrain Epoch 11/15 | Loss: 0.5137
Pretrain Epoch 12/15 | Loss: 0.5179
Pretrain Epoch 13/15 | Loss: 0.5020
Pretrain Epoch 14/15 | Loss: 0.5087
Pretrain Epoch 15/15 | Loss: 0.5229
Pretrained encoder weights saved.
/content


## Section 6: Seed Synchronization & In-Place train.py Patch
Modifies the standard train.py parser to add seed arguments and lock determinism for pytorch.


In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet

# Read and modify train.py to add --seed argument and PyTorch reproducibility settings
with open('train.py', 'r') as f:
    content = f.read()

# Add argument to parser
if "--seed" not in content:
    content = content.replace(
        "parser.add_argument('--num_workers', default=4, type=int)",
        "parser.add_argument('--num_workers', default=4, type=int)\n    parser.add_argument('--seed', default=42, type=int, help='random seed')"
    )

    # Sort glob and remove data split validation leakage
    if "img_ids = sorted(img_ids)" not in content:
        content = content.replace(
            "    train_img_ids, val_img_ids = train_test_split(img_ids, test_size=0.3, random_state=41)\n    val_img_ids, test_img_ids = train_test_split(img_ids, test_size=0.5, random_state=41)",
            "    img_ids = sorted(img_ids)\n    train_img_ids, val_img_ids = train_test_split(img_ids, test_size=0.2, random_state=41)"
        )

    seed_code = """
    # Set random seed
    import random
    import numpy as np
    import torch
    seed = config['seed']
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
"""
    content = content.replace(
        "config = vars(parse_args())",
        "config = vars(parse_args())" + seed_code
    )

    with open('train.py', 'w') as f:
        f.write(content)
    print("train.py modified successfully with seed parameters.")
else:
    print("train.py already has seed options.")

%cd /content


/content/Auto-Generate-WLs/pytorch-nested-unet
train.py modified successfully with seed parameters.
/content


## Section 7: Train Baseline Segmentor on Gold Standard Train Set (20 images)


In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
# Train on the 20 GS images split with seed 42
!python train.py --dataset busi-gs-25 --arch NestedUNet \
    --img_ext .png --mask_ext .png --batch_size 4 --input_w 256 --input_h 256 --seed 42 --optimizer Adam --lr 3e-4 --epochs 200



/content/Auto-Generate-WLs/pytorch-nested-unet
--------------------
name: busi-gs-25_NestedUNet_256_woDS
epochs: 200
batch_size: 4
arch: NestedUNet
deep_supervision: False
input_channels: 3
num_classes: 1
input_w: 256
input_h: 256
loss: BCEDiceLoss
dataset: busi-gs-25
img_ext: .png
mask_ext: .png
optimizer: Adam
lr: 0.0003
momentum: 0.9
weight_decay: 0.0001
nesterov: False
scheduler: CosineAnnealingLR
min_lr: 1e-05
factor: 0.1
patience: 2
milestones: 1,2
gamma: 0.6666666666666666
early_stopping: -1
num_workers: 4
seed: 42
--------------------
=> creating model NestedUNet
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freez

## Section 8: Evaluate Baseline Model on reserved `busi-test` (5 images)


In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
!python eval.py --name busi-gs-25_NestedUNet_256_woDS --dataset busi-test-130
%cd /content


/content/Auto-Generate-WLs/pytorch-nested-unet
--------------------
arch: NestedUNet
batch_size: 4
dataset: busi-gs-25
deep_supervision: False
early_stopping: -1
epochs: 200
factor: 0.1
gamma: 0.6666666666666666
img_ext: .png
input_channels: 3
input_h: 256
input_w: 256
loss: BCEDiceLoss
lr: 0.0003
mask_ext: .png
milestones: 1,2
min_lr: 1e-05
momentum: 0.9
name: busi-gs-25_NestedUNet_256_woDS
nesterov: False
num_classes: 1
num_workers: 4
optimizer: Adam
patience: 2
scheduler: CosineAnnealingLR
seed: 42
weight_decay: 0.0001
--------------------
=> creating model NestedUNet
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freez

## Section 9: Pre-Coarse Complexity Gate (Idea 1)
Trains a Random Forest classifier on the GS images using visual complexity features to predict if MedSAM will segment them correctly.


In [ ]:
# Write complexity_gate.py in generate-weak-labels/
complexity_gate_src = """
import cv2
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

def compute_complexity_score(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    img = cv2.resize(img, (256, 256))
    img_norm = img / 255.0

    local_var = float(np.std(img_norm))
    edges = cv2.Canny(img, 50, 150)
    edge_density = float(np.sum(edges > 0) / (256 * 256))

    hist, _ = np.histogram(img_norm, bins=32, density=True)
    hist = hist + 1e-10
    entropy = float(-np.sum(hist * np.log(hist)))

    border = np.concatenate([img_norm[0, :], img_norm[-1, :], img_norm[:, 0], img_norm[:, -1]])
    bg_homogeneity = float(1.0 / (np.std(border) + 1e-5))

    return {
        'local_var': local_var,
        'edge_density': edge_density,
        'entropy': entropy,
        'bg_homogeneity': bg_homogeneity
    }

def train_complexity_classifier(gs_images, gs_masks, medsam_results):
    X, y = [], []
    for img_path, mask_path, pred_mask in zip(gs_images, gs_masks, medsam_results):
        features = compute_complexity_score(img_path)
        X.append(list(features.values()))

        gt = cv2.imread(mask_path, 0)
        gt = (gt > 127).astype(np.uint8)
        pred = (pred_mask > 127).astype(np.uint8)

        intersection = np.logical_and(gt, pred).sum()
        dice = (2. * intersection) / (gt.sum() + pred.sum() + 1e-6)
        y.append(1 if dice > 0.5 else 0)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X_scaled, y)
    return clf, scaler

def gate_images(image_paths, clf, scaler, threshold=0.5):
    approved, rejected = [], []
    for path in image_paths:
        features = compute_complexity_score(path)
        X_feat = scaler.transform([list(features.values())])
        if len(clf.classes_) < 2:
            prob = 1.0 if clf.classes_[0] == 1 else 0.0
        else:
            prob = clf.predict_proba(X_feat)[0][1]
        if prob >= threshold:
            approved.append(path)
        else:
            rejected.append(path)

    if len(approved) < 40:
        probs = []
        for path in image_paths:
            features = compute_complexity_score(path)
            X_feat = scaler.transform([list(features.values())])
            if len(clf.classes_) < 2:
                prob = 1.0
            else:
                prob = clf.predict_proba(X_feat)[0][1]
            probs.append((prob, path))
        probs.sort(key=lambda x: x[0], reverse=True)
        approved = [path for _, path in probs[:40]]
        rejected = [path for _, path in probs[40:]]
    return approved, rejected
"""
os.makedirs("Auto-Generate-WLs/generate-weak-labels", exist_ok=True)
with open("Auto-Generate-WLs/generate-weak-labels/complexity_gate.py", "w") as f:
    f.write(complexity_gate_src)
print("complexity_gate.py created successfully.")


complexity_gate.py created successfully.


## Section 10: Augmentation Consistency Check (TTA) (Idea 3)
Runs augmentations on coarse model predictions and filters out images with low alignment consistency.


In [ ]:
import torch
import torchvision.transforms.functional as TF
import numpy as np
import cv2
from scipy import ndimage

def augmentation_consistency_check(image, coarse_model, device, n_augments=8, iou_threshold=0.20):
    # Ensure image is 3 channels (RGB)
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

    H, W = image.shape[:2]
    angles = np.linspace(-30, 30, n_augments)
    boxes = []

    coarse_model.eval()
    for angle in angles:
        # Scale to [0.0, 1.0] and transpose to (C, H, W) to match 3-channel baseline model expectation
        img_tensor = torch.tensor(image.transpose(2,0,1), dtype=torch.float32).unsqueeze(0) / 255.0
        aug_tensor = TF.rotate(img_tensor, float(angle))
        aug_tensor_resized = torch.nn.functional.interpolate(aug_tensor, size=(256, 256), mode='bilinear', align_corners=True)

        with torch.no_grad():
            pred = coarse_model(aug_tensor_resized.to(device))
            if isinstance(pred, list):
                pred = pred[-1]
            pred = torch.sigmoid(pred).squeeze().cpu().numpy()

        pred_resized = cv2.resize(pred, (W, H))
        pred_bin = (pred_resized > 0.5).astype(np.uint8)

        pred_back = ndimage.rotate(pred_bin, -float(angle), reshape=False, order=0)
        pred_back = (pred_back > 0.5).astype(np.uint8)

        contours, _ = cv2.findContours(pred_back, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
            boxes.append([x, y, x + w, y + h])

    if len(boxes) < n_augments // 4:
        return False, 0.0

    ious = []
    for i in range(len(boxes)):
        for j in range(i+1, len(boxes)):
            b1, b2 = boxes[i], boxes[j]
            inter_x1 = max(b1[0], b2[0])
            inter_y1 = max(b1[1], b2[1])
            inter_x2 = min(b1[2], b2[2])
            inter_y2 = min(b1[3], b2[3])

            inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
            b1_area = (b1[2] - b1[0]) * (b1[3] - b1[1])
            b2_area = (b2[2] - b2[0]) * (b2[3] - b2[1])
            union_area = b1_area + b2_area - inter_area

            ious.append(inter_area / (union_area + 1e-5))

    mean_iou = float(np.mean(ious)) if ious else 0.0
    return (mean_iou >= iou_threshold), mean_iou

## Section 11: Setup MedSAM Model & Inference with Centroids (Idea 5 - Part 1)
Prepares inference helper methods for MedSAM using bounding boxes combined with centroids.


In [ ]:
import torch
import cv2
import numpy as np
from segment_anything import sam_model_registry
from PIL import Image as PILImage
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
medsam_checkpoint = "/content/drive/MyDrive/MEDSAM/medsam_vit_b.pth"

# Load MedSAM Model
assert os.path.exists(medsam_checkpoint), "Place medsam_vit_b.pth in Google Drive first!"
medsam_model = sam_model_registry["vit_b"](checkpoint=medsam_checkpoint)
medsam_model.to(device)
medsam_model.eval()
print("MedSAM loaded successfully onto", device)

def run_medsam_refinement(image, coarse_mask):
    # Refines coarse_mask using MedSAM with bounding box and centroid point prompts.
    H, W = image.shape[:2]
    num_labels, labels = cv2.connectedComponents(coarse_mask)
    if num_labels <= 1:
        return None

    largest_label = 1 + np.argmax([np.sum(labels == i) for i in range(1, num_labels)])
    clean_mask = (labels == largest_label).astype(np.uint8)

    ys, xs = np.where(clean_mask > 0)
    x1, y1 = xs.min(), ys.min()
    x2, y2 = xs.max(), ys.max()

    # Expand the bounding box to give MedSAM surrounding context (Correct padding)
    pad = 10
    x1 = max(x1 - pad, 0)
    y1 = max(y1 - pad, 0)
    x2 = min(x2 + pad, W)
    y2 = min(y2 + pad, H)

    # Extract Centroid
    cx = int(np.mean(xs))
    cy = int(np.mean(ys))

    # Scale inputs to 1024x1024
    box_1024 = [x1 * 1024 / W, y1 * 1024 / H, x2 * 1024 / W, y2 * 1024 / H]

    # Preprocess Image with BGR to RGB conversion (MedSAM expects RGB)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_1024 = cv2.resize(image_rgb, (1024, 1024)).astype(np.float32)
    img_1024 = (img_1024 - img_1024.min()) / np.clip(
        img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
    )
    img_1024 = np.transpose(img_1024, (2, 0, 1))
    img_tensor = torch.tensor(img_1024).unsqueeze(0).to(device)

    with torch.no_grad():
        image_embedding = medsam_model.image_encoder(img_tensor)

        box_torch = torch.tensor(box_1024, dtype=torch.float32).to(device).unsqueeze(0).unsqueeze(0)

        # MedSAM is trained on box prompts only (points=None)
        sparse_emb, dense_emb = medsam_model.prompt_encoder(
            points=None, boxes=box_torch, masks=None)

        low_res_logits, _ = medsam_model.mask_decoder(
            image_embeddings=image_embedding,
            image_pe=medsam_model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_emb,
            dense_prompt_embeddings=dense_emb,
            multimask_output=False)

    probs = torch.sigmoid(low_res_logits).cpu().numpy().squeeze()
    probs = cv2.resize(probs, (W, H))
    probs = np.clip(probs, 0, 1)
    refined_mask = (probs > 0.5).astype(np.uint8) * 255
    return refined_mask



MedSAM loaded successfully onto cuda


## Section 12: V2 Morphological Gate Filters (Idea 2)
Implement geometry-based screening including aspect, convexity, stripe boundary detection, and components analysis.


In [ ]:
def morphological_gate_v2(coarse_mask, image_shape=(256, 256)):
    mask = (coarse_mask > 127).astype(np.uint8)
    H, W = mask.shape[:2]

    # Filter 1: Extreme fill ratio
    fill_ratio = (mask > 0).mean()
    if fill_ratio > 0.97 or fill_ratio < 0.002:   # Lower floor from 0.005 to 0.002
        return False, f"extreme_fill ({fill_ratio:.3f})", None

    # Filter 2: Fragment check
    num_labels, labels = cv2.connectedComponents(mask)
    if num_labels <= 1:
        return False, "no_object", None

    component_sizes = [np.sum(labels == i) for i in range(1, num_labels)]
    significant_sizes = [s for s in component_sizes if s >= 50]   # Lowered from 100 to 50

    if len(significant_sizes) > 200:   # Raised from 100 to 200 to allow more complex/split cases
        return False, f"too_fragmented ({len(significant_sizes)})", None

    # Keep largest component
    largest_label = 1 + np.argmax(component_sizes)
    clean_mask = (labels == largest_label).astype(np.uint8)

    # Filter 3: Ambiguous dual-lesion check (among significant components)
    sorted_sig_sizes = sorted(significant_sizes, reverse=True)
    if len(sorted_sig_sizes) >= 2:
        size_ratio = sorted_sig_sizes[1] / (sorted_sig_sizes[0] + 1e-5)
        if size_ratio > 0.75:   # Raised from 0.6 to 0.75
            return False, f"ambiguous_components ({size_ratio:.2f})", None

    # Filter 4: Contour checks
    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False, "no_contour", None

    main_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(main_contour)

    # Aspect Check
    aspect_ratio = max(w, h) / (min(w, h) + 1e-5)
    if aspect_ratio > 15.0:   # Raised from 12.0 to 15.0
        return False, f"bad_aspect ({aspect_ratio:.1f})", None

    # Size Gate
    area = cv2.contourArea(main_contour)
    if area < 0.0005 * H * W:   # Lowered from 0.001 to 0.0005 to keep smaller lesions
        return False, f"too_small ({area:.0f})", None

    # Convexity Gate
    hull = cv2.convexHull(main_contour)
    hull_area = cv2.contourArea(hull)
    convexity = area / (hull_area + 1e-5)
    if convexity < 0.03:   # Lowered from 0.05 to 0.03
        return False, f"low_convexity ({convexity:.2f})", None

    # Filter 5: Stripe artifact detection
    ys, xs = np.where(clean_mask > 0)
    if len(xs) > 0:
        unique_cols = len(np.unique(xs))
        unique_rows = len(np.unique(ys))
        stripe_ratio = unique_cols / (unique_rows + 1e-5)
        if stripe_ratio > 7.0:   # Raised from 5.0 to 7.0
            return False, f"stripe_artifact ({stripe_ratio:.1f})", None

    return True, "passed", clean_mask * 255


## Section 13: Radiomics Quality Predictor (Idea 7)
Grades weak label segments using texture/geometric properties. Utilizes a scikit-image fallback.


In [ ]:
from skimage.measure import regionprops
from skimage.feature import graycomatrix, graycoprops
from sklearn.ensemble import GradientBoostingRegressor

def extract_skimage_features(image, mask):
    mask_bin = (mask > 127).astype(np.uint8)
    props = regionprops(mask_bin)
    if len(props) == 0:
        return {'area': 0.0, 'eccentricity': 0.0, 'perimeter': 0.0, 'mean_intensity': 0.0, 'contrast': 0.0}

    prop = max(props, key=lambda x: x.area)

    # Convert image to uint8 scale for graycomatrix
    img_clipped = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
    glcm = graycomatrix(img_clipped, distances=[2], angles=[0], levels=256, symmetric=True, normed=True)
    contrast = float(graycoprops(glcm, 'contrast')[0, 0])

    return {
        'area': float(prop.area),
        'eccentricity': float(prop.eccentricity),
        'perimeter': float(prop.perimeter),
        'mean_intensity': float(np.mean(image[mask_bin > 0]) if mask_bin.sum() > 0 else 0),
        'contrast': contrast
    }

def train_radiomics_quality_model(gs_image_paths, gs_mask_paths, medsam_predictions):
    X, y = [], []
    for img_p, gt_p, pred_mask in zip(gs_image_paths, gs_mask_paths, medsam_predictions):
        img = cv2.imread(img_p, cv2.IMREAD_GRAYSCALE)
        gt = (cv2.imread(gt_p, cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)

        feats = extract_skimage_features(img, pred_mask)
        X.append(list(feats.values()))

        intersection = np.logical_and(gt, (pred_mask > 127)).sum()
        dice = (2. * intersection) / (gt.sum() + (pred_mask > 127).sum() + 1e-5)
        y.append(dice)

    model = GradientBoostingRegressor(n_estimators=30, random_state=42)
    model.fit(X, y)
    return model


## Section 14: Curriculum sorting & Multi-Gate Generation Loop (Idea 5)
1. Runs Complexity Gating on unlabeled images.
2. Performs TTA consistency checks on remaining images.
3. Refines predictions using Centroid-MedSAM.
4. Filters refined weak labels through the V2 Morphological Gate and Radiomics Regressor.
5. Saves successful weak labels for dataset augmentation.


In [ ]:
import sys
sys.path.append("Auto-Generate-WLs/generate-weak-labels")
import complexity_gate
import importlib
importlib.reload(complexity_gate)
from complexity_gate import compute_complexity_score, train_complexity_classifier, gate_images

# Load Trained Coarse Model for Predictions
import torch
import archs

# Point to baseline model trained on busi-gs-25
coarse_model_path = "Auto-Generate-WLs/pytorch-nested-unet/models/busi-gs-25_NestedUNet_256_woDS/model.pth"
coarse_net = archs.NestedUNet(num_classes=1, input_channels=3, deep_supervision=False).to(device)
coarse_net.load_state_dict(torch.load(coarse_model_path, map_location=device))
coarse_net.eval()
print("Coarse prediction model loaded.")

# Prepare paths
unlabeled_img_dir = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-unlabeled/images"

# Exclude GS train and test images to prevent redundancy and data leakage
gs_train_files = set(os.listdir("Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-gs-25/images"))
gs_test_files = set(os.listdir("Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-test-130/images"))
exclude_files = gs_train_files.union(gs_test_files)

unlabeled_files = sorted([
    f for f in os.listdir(unlabeled_img_dir)
    if f.endswith('.png') and f not in exclude_files
])

print(f"Total candidate unlabeled images to process: {len(unlabeled_files)}")

# Pre-train complexity gate on training GS samples (25 samples)
gs_img_paths = [os.path.join("Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-gs-25/images", f) for f in os.listdir("Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-gs-25/images")]
gs_mask_paths = [os.path.join("Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-gs-25/masks/0", f) for f in os.listdir("Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-gs-25/images")]

# Generate actual MedSAM predictions on GS training set for radiomics training
dummy_medsam_preds = []
for img_p in gs_img_paths:
    img = cv2.imread(img_p)
    H, W = img.shape[:2]
    t_img = torch.tensor(img.transpose(2,0,1)).unsqueeze(0).float().to(device) / 255.0
    t_img = torch.nn.functional.interpolate(t_img, size=(256, 256), mode='bilinear')
    with torch.no_grad():
        c_pred = torch.sigmoid(coarse_net(t_img)).squeeze().cpu().numpy()
    coarse_mask = (cv2.resize(c_pred, (W, H)) > 0.5).astype(np.uint8) * 255
    refined = run_medsam_refinement(img, coarse_mask)
    if refined is None:
        refined = coarse_mask
    dummy_medsam_preds.append(refined)

# Train Complexity gate and Radiomics Quality predictor
from complexity_gate import train_complexity_classifier, gate_images
clf_comp, scaler_comp = train_complexity_classifier(gs_img_paths, gs_mask_paths, dummy_medsam_preds)
quality_regressor = train_radiomics_quality_model(gs_img_paths, gs_mask_paths, dummy_medsam_preds)

print("Trained gate models successfully.")

# STEP 1: Predict Coarse Masks & Get Confidence for Curriculum Sort
coarse_masks_dict = {}
unlabeled_paths = []
coarse_preds = []

for fname in unlabeled_files:
    img_path = os.path.join(unlabeled_img_dir, fname)
    image = cv2.imread(img_path)
    H, W = image.shape[:2]

    t_img = torch.tensor(image.transpose(2,0,1)).unsqueeze(0).float().to(device) / 255.0
    t_img = torch.nn.functional.interpolate(t_img, size=(256, 256), mode='bilinear')
    with torch.no_grad():
        c_pred = torch.sigmoid(coarse_net(t_img)).squeeze().cpu().numpy()

    coarse_masks_dict[fname] = (cv2.resize(c_pred, (W, H)) > 0.5).astype(np.uint8) * 255
    unlabeled_paths.append(img_path)
    coarse_preds.append(c_pred)

# Curriculum Sort (Strategy: easy_first)
def curriculum_sort(image_paths, coarse_predictions, strategy='easy_first'):
    confidence_scores = []
    for pred in coarse_predictions:
        max_prob = pred.max()
        mask = (pred > 0.5).astype(float)
        size_penalty = abs(mask.mean() - 0.15)
        score = max_prob - size_penalty
        confidence_scores.append(score)
    sorted_pairs = sorted(zip(confidence_scores, image_paths), key=lambda x: x[0], reverse=(strategy == 'easy_first'))
    return [path for _, path in sorted_pairs], [score for score, _ in sorted_pairs]

sorted_paths, _ = curriculum_sort(unlabeled_paths, coarse_preds, strategy='easy_first')

# Output directories
save_wl_mask_dir = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-25-wl-v2/masks/0"
save_wl_img_dir  = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-25-wl-v2/images"
os.makedirs(save_wl_mask_dir, exist_ok=True)
os.makedirs(save_wl_img_dir, exist_ok=True)

# ─── KEY FIX: Cap at 100 weak labels to avoid noise overfitting ───
MAX_WEAK_LABELS = 100

# PROCESS GENERATION LOOP IN CURRICULUM ORDER
saved_wl_names = []
skipped_counts = {}

# Complexity Gating step
approved_paths, rejected_paths = gate_images(sorted_paths, clf_comp, scaler_comp, threshold=0.15)
print(f"Complexity Gate: Approved {len(approved_paths)} images, Rejected {len(rejected_paths)} images.")

for img_path in approved_paths:
    # ─── STOP early once we hit the cap ───
    if len(saved_wl_names) >= MAX_WEAK_LABELS:
        print(f"✅ Reached cap of {MAX_WEAK_LABELS} weak labels. Stopping early.")
        break

    fname = os.path.basename(img_path)
    image = cv2.imread(img_path)
    coarse_mask = coarse_masks_dict[fname]

    # Gate 2: Consistency Check
    is_consistent, iou_val = augmentation_consistency_check(image, coarse_net, device, n_augments=6, iou_threshold=0.20)
    if not is_consistent:
        skipped_counts[fname] = f"inconsistent_TTA ({iou_val:.2f})"
        continue

    # Refine using MedSAM with Centroids
    refined_mask = run_medsam_refinement(image, coarse_mask)
    if refined_mask is None:
        skipped_counts[fname] = "medsam_failed"
        continue

    # Gate 3: Morphological Gate V2
    passed_morph, morph_reason, cleaned_mask = morphological_gate_v2(refined_mask)
    if not passed_morph:
        skipped_counts[fname] = f"morph_{morph_reason}"
        continue

    # Gate 4: Radiomics Quality Check
    feats = extract_skimage_features(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), refined_mask)
    pred_quality = quality_regressor.predict([list(feats.values())])[0]
    if pred_quality < 0.03:
        skipped_counts[fname] = f"low_quality_score ({pred_quality:.2f})"
        continue

    # Save weak label
    cv2.imwrite(os.path.join(save_wl_mask_dir, fname), cleaned_mask)
    shutil.copy(img_path, os.path.join(save_wl_img_dir, fname))
    saved_wl_names.append(fname)

print(f"Loop finished. Saved {len(saved_wl_names)} weak labels.")
print("Skipped reasons count:")
from collections import Counter
print(Counter(skipped_counts.values()))


Coarse prediction model loaded.
Total candidate unlabeled images to process: 493
Trained gate models successfully.
Complexity Gate: Approved 165 images, Rejected 328 images.
Loop finished. Saved 33 weak labels.
Skipped reasons count:
Counter({'low_quality_score (0.01)': 18, 'low_quality_score (0.00)': 12, 'inconsistent_TTA (0.19)': 11, 'inconsistent_TTA (0.17)': 11, 'inconsistent_TTA (0.15)': 9, 'inconsistent_TTA (0.18)': 9, 'inconsistent_TTA (0.13)': 7, 'inconsistent_TTA (0.10)': 6, 'inconsistent_TTA (0.08)': 6, 'inconsistent_TTA (0.14)': 5, 'low_quality_score (-0.00)': 5, 'inconsistent_TTA (0.16)': 5, 'morph_extreme_fill (0.002)': 5, 'morph_extreme_fill (0.001)': 5, 'low_quality_score (0.02)': 3, 'inconsistent_TTA (0.07)': 3, 'inconsistent_TTA (0.09)': 3, 'inconsistent_TTA (0.05)': 2, 'inconsistent_TTA (0.12)': 2, 'inconsistent_TTA (0.20)': 2, 'low_quality_score (0.03)': 1, 'inconsistent_TTA (0.06)': 1, 'morph_bad_aspect (18.2)': 1})


In [ ]:
# DIAGNOSTIC: Count how many weak labels were successfully generated
import os

wl_mask_dir = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-25-wl-v2/masks/0"
wl_img_dir  = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-25-wl-v2/images"

if os.path.exists(wl_mask_dir):
    n_wl = len(os.listdir(wl_mask_dir))
    print(f"✅ Weak labels generated: {n_wl}")
else:
    print("❌ No weak labels directory found — generation failed!")

aug_img_dir = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-25-augmented-v2/images"
if os.path.exists(aug_img_dir):
    total = len(os.listdir(aug_img_dir))
    print(f"📦 Total augmented training samples (GS + WL): {total}")

✅ Weak labels generated: 33


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Section 15: Distance-Weighted Loss Retraining (Idea 6)
We patch `losses.py` and `train.py` to support distance-weighted BCE + Dice loss on weak-labeled training samples.


In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet

import os
# Force reset files to clean git state first to ensure clean patching
os.system("git checkout -- train.py dataset.py losses.py")

# ── 1. Modify dataset.py to return is_weak flag ───────────────────────────────
with open('dataset.py', 'r') as f:
    ds_content = f.read()

ds_content = ds_content.replace('\r\n', '\n')
if "is_weak" not in ds_content:
    ds_content = ds_content.replace(
        "return img, mask, {'img_id': img_id}",
        "is_weak = 1.0 if 'wl' in img_id or 'weak' in img_id else 0.0\n        return img, mask, {'img_id': img_id, 'is_weak': is_weak}"
    )
    with open('dataset.py', 'w') as f:
        f.write(ds_content)
    print("dataset.py modified.")

# ── 2. Add DistanceWeightedLoss and DiceLoss to losses.py ────────────────────
losses_code = """
import torch
import torch.nn as nn
from scipy.ndimage import distance_transform_edt
import numpy as np

class DistanceWeightedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce_loss_raw = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, output, target, is_weak_batch=None):
        if is_weak_batch is None or not is_weak_batch.any():
            return self.bce_loss_raw(output, target).mean()

        batch_weights = []
        for i in range(target.shape[0]):
            if is_weak_batch[i] > 0.5:
                mask_np = target[i, 0].detach().cpu().numpy() > 0.5
                if mask_np.sum() > 0:
                    dist_in = distance_transform_edt(mask_np)
                    dist_out = distance_transform_edt(1.0 - mask_np)
                    dist_to_boundary = dist_in + dist_out
                    if dist_to_boundary.max() > 0:
                        dist_norm = dist_to_boundary / dist_to_boundary.max()
                    else:
                        dist_norm = dist_to_boundary
                    weight = 0.4 + 0.6 * dist_norm
                else:
                    weight = np.ones_like(mask_np, dtype=np.float32)
                batch_weights.append(torch.tensor(weight, dtype=torch.float32).unsqueeze(0))
            else:
                batch_weights.append(torch.ones(1, *target.shape[2:], dtype=torch.float32))

        pixel_weights = torch.stack(batch_weights).to(output.device)
        bce_map = self.bce_loss_raw(output, target)
        weighted_bce = bce_map * pixel_weights
        if is_weak_batch is not None:
            image_weights = (1.0 - 0.5 * is_weak_batch).view(-1, 1, 1, 1)
            weighted_bce = weighted_bce * image_weights
        return weighted_bce.mean()

class DiceLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input, target, is_weak_batch=None):
        smooth = 1e-5
        input = torch.sigmoid(input)
        num = target.size(0)
        input = input.view(num, -1)
        target = target.view(num, -1)
        intersection = (input * target)
        dice = (2. * intersection.sum(1) + smooth) / (input.sum(1) + target.sum(1) + smooth)
        loss_per_image = 1 - dice
        if is_weak_batch is not None:
            weights = 1.0 - 0.5 * is_weak_batch
            return (loss_per_image * weights).mean()
        return loss_per_image.mean()
"""

with open('losses.py', 'a') as f:
    f.write(losses_code)
print("losses.py modified with DistanceWeightedLoss and DiceLoss.")

# ── 3. Patch train.py: seed + val_dataset flag + DWL integration ─────────────
with open('train.py', 'r') as f:
    tr_content = f.read()

tr_content = tr_content.replace('\r\n', '\n')

# Add --seed and --val_dataset arguments to parser
if "--seed" not in tr_content:
    tr_content = tr_content.replace(
        "parser.add_argument('--num_workers', default=4, type=int)",
        "parser.add_argument('--num_workers', default=4, type=int)\n    parser.add_argument('--seed', default=42, type=int, help='random seed')\n    parser.add_argument('--val_dataset', default=None, type=str, help='separate clean validation dataset folder')"
    )
    seed_code = """
    # Set random seed for reproducibility
    import random
    import numpy as np
    import torch
    seed = config['seed']
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
"""
    tr_content = tr_content.replace(
        "config = vars(parse_args())",
        "config = vars(parse_args())" + seed_code
    )

# ── KEY FIX: Support separate clean validation dataset ───────────────────────
if "val_dataset" not in tr_content:
    tr_content = tr_content.replace(
        "    img_ids = sorted(img_ids)\n    train_img_ids, val_img_ids = train_test_split(img_ids, test_size=0.2, random_state=41)",
        """    img_ids = sorted(img_ids)
    if config.get('val_dataset'):
        # Use a separate clean validation folder (GS-only) — prevents noisy WL
        # from corrupting model selection
        train_img_ids = img_ids
        val_img_ids = sorted([
            os.path.splitext(fn)[0]
            for fn in os.listdir(os.path.join('inputs', config['val_dataset'], 'images'))
            if fn.endswith('.png')
        ])
        print(f"[val_dataset] Train: {len(train_img_ids)} | Val (clean GS): {len(val_img_ids)}")
    else:
        train_img_ids, val_img_ids = train_test_split(img_ids, test_size=0.2, random_state=41)"""
    )
else:
    print("val_dataset support already in train.py.")

# ── Integrate DistanceWeightedLoss ────────────────────────────────────────────
if "DistanceWeightedLoss" not in tr_content:
    tr_content = tr_content.replace(
        "import losses",
        "import losses\nfrom losses import DistanceWeightedLoss\ndw_loss_fn = DistanceWeightedLoss()"
    ).replace(
        "for input, target, _ in train_loader:",
        "for input, target, meta in train_loader:"
    ).replace(
        "        else:\n            output = model(input)\n            loss = criterion(output, target)",
        "        else:\n            output = model(input)\n            is_weak_batch = meta.get('is_weak', torch.zeros(target.shape[0])).to(target.device)\n            bce_loss = dw_loss_fn(output, target, is_weak_batch)\n            dice_loss = losses.DiceLoss()(output, target, is_weak_batch)\n            loss = dice_loss + 0.5 * bce_loss"
    )

with open('train.py', 'w') as f:
    f.write(tr_content)
print("train.py patched with: seed, val_dataset flag, DistanceWeightedLoss.")

%cd /content


/content/Auto-Generate-WLs/pytorch-nested-unet
dataset.py modified.
losses.py modified with DistanceWeightedLoss and DiceLoss.
val_dataset support already in train.py.
train.py patched with: seed, val_dataset flag, DistanceWeightedLoss.
/content


## Section 16: Build Augmented-V2 Dataset & Retrain Model
Combines the 20 GS training images with the generated V2 weak labels, and runs training.


In [ ]:
import os, shutil

# ─── Directory definitions ───────────────────────────────────────────────────
gs_img_dir   = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-gs-25/images"
gs_mask_dir  = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-gs-25/masks/0"
wl_img_dir   = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-25-wl-v2/images"
wl_mask_dir  = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-25-wl-v2/masks/0"

# Training: 20 GS + 100 WL
aug_train_img  = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-aug-train/images"
aug_train_mask = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-aug-train/masks/0"

# Validation: 5 GS only — CLEAN SIGNAL for model selection
aug_val_img    = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-aug-val/images"
aug_val_mask   = "Auto-Generate-WLs/pytorch-nested-unet/inputs/busi-aug-val/masks/0"

# ─── Clean slate rebuild ─────────────────────────────────────────────────────
for d in [aug_train_img, aug_train_mask, aug_val_img, aug_val_mask]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# ─── Split 25 GS images: 20 train / 5 val (deterministic by sorted name) ─────
gs_files = sorted(os.listdir(gs_img_dir))
gs_val_files   = gs_files[:5]    # 5 clean GS for validation
gs_train_files = gs_files[5:]    # 20 GS for training

# Copy GS train (20 images)
for f in gs_train_files:
    shutil.copy(os.path.join(gs_img_dir,  f), os.path.join(aug_train_img,  f))
    shutil.copy(os.path.join(gs_mask_dir, f), os.path.join(aug_train_mask, f))

# Copy GS val (5 images) — GS ONLY, no weak labels in validation
for f in gs_val_files:
    shutil.copy(os.path.join(gs_img_dir,  f), os.path.join(aug_val_img,  f))
    shutil.copy(os.path.join(gs_mask_dir, f), os.path.join(aug_val_mask, f))

# Copy 100 weak labels — training set ONLY
wl_files = sorted(os.listdir(wl_mask_dir))
for f in wl_files:
    shutil.copy(os.path.join(wl_img_dir,  f), os.path.join(aug_train_img,  f"wl_{f}"))
    shutil.copy(os.path.join(wl_mask_dir, f), os.path.join(aug_train_mask, f"wl_{f}"))

print(f"Training set : {len(os.listdir(aug_train_img))} images (20 GS + {len(wl_files)} WL)")
print(f"Validation set: {len(os.listdir(aug_val_img))} images (5 clean GS — zero weak labels)")

# ─── Train with clean validation signal ──────────────────────────────────────
%cd /content/Auto-Generate-WLs/pytorch-nested-unet
!python train.py \
    --dataset busi-aug-train \
    --val_dataset busi-aug-val \
    --arch NestedUNet \
    --img_ext .png --mask_ext .png \
    --batch_size 4 --input_w 256 --input_h 256 \
    --seed 42 --optimizer Adam --lr 3e-4 --epochs 200
%cd /content

Training set : 53 images (20 GS + 33 WL)
Validation set: 5 images (5 clean GS — zero weak labels)
/content/Auto-Generate-WLs/pytorch-nested-unet
--------------------
name: busi-aug-train_NestedUNet_256_woDS
epochs: 200
batch_size: 4
arch: NestedUNet
deep_supervision: False
input_channels: 3
num_classes: 1
input_w: 256
input_h: 256
loss: BCEDiceLoss
dataset: busi-aug-train
img_ext: .png
mask_ext: .png
optimizer: Adam
lr: 0.0003
momentum: 0.9
weight_decay: 0.0001
nesterov: False
scheduler: CosineAnnealingLR
min_lr: 1e-05
factor: 0.1
patience: 2
milestones: 1,2
gamma: 0.6666666666666666
early_stopping: -1
num_workers: 4
seed: 42
val_dataset: busi-aug-val
--------------------
=> creating model NestedUNet
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that e

## Section 17: Final Evaluation & Comparisons
Evaluates the final augmented-v2 model and prints baseline vs improved results on the reserved `busi-test` set.


In [ ]:
%cd /content/Auto-Generate-WLs/pytorch-nested-unet

print("\n" + "="*55)
print("EVALUATION: UNSEEN TEST SET (busi-test-130, 130 images)")
print("="*55)

print("\n--- 1. BASELINE (25 GS only) ---")
!python eval.py --name busi-gs-25_NestedUNet_256_woDS --dataset busi-test-130

print("\n--- 2. IMPROVED (20 GS train + 100 WL, clean val) ---")
!python eval.py --name busi-aug-train_NestedUNet_256_woDS --dataset busi-test-130

%cd /content


/content/Auto-Generate-WLs/pytorch-nested-unet

EVALUATION: UNSEEN TEST SET (busi-test-130, 130 images)

--- 1. BASELINE (25 GS only) ---
--------------------
arch: NestedUNet
batch_size: 4
dataset: busi-gs-25
deep_supervision: False
early_stopping: -1
epochs: 200
factor: 0.1
gamma: 0.6666666666666666
img_ext: .png
input_channels: 3
input_h: 256
input_w: 256
loss: BCEDiceLoss
lr: 0.0003
mask_ext: .png
milestones: 1,2
min_lr: 1e-05
momentum: 0.9
name: busi-gs-25_NestedUNet_256_woDS
nesterov: False
num_classes: 1
num_workers: 4
optimizer: Adam
patience: 2
scheduler: CosineAnnealingLR
seed: 42
weight_decay: 0.0001
--------------------
=> creating model NestedUNet
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLo

## Section 18: Test-Time Augmentation (TTA) Evaluation
We implement an ensemble evaluation using Test-Time Augmentation (4 flip configurations) to improve predictions on the test set.

In [ ]:
# ==============================================================================
# ## Section 18: Test-Time Augmentation (TTA) Evaluation (FIXED)
# ==============================================================================
%cd /content/Auto-Generate-WLs/pytorch-nested-unet

eval_tta_code = """
import argparse
import os
from glob import glob
import cv2
import numpy as np
import torch
import torch.backends.cudnn as cudnn
import yaml
from albumentations.augmentations import transforms
from albumentations.core.composition import Compose
from tqdm import tqdm

import archs
from dataset import Dataset

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--name', default=None, required=True, help='model name')
    parser.add_argument('--dataset', default=None, required=True, help='dataset name')
    parser.add_argument('--tta_mode', default='horizontal', choices=['none', 'horizontal', 'all'], help='TTA mode')
    return vars(parser.parse_args())

def main():
    args = parse_args()
    with open('models/%s/config.yml' % args['name'], 'r') as f:
        config = yaml.load(f, Loader=yaml.FullLoader)

    cudnn.benchmark = True

    model = archs.__dict__[config['arch']](
        num_classes=config['num_classes'],
        input_channels=config['input_channels'],
        deep_supervision=config['deep_supervision']
    )
    model = model.cuda()

    model_path = os.path.join('models', args['name'], 'model.pth')
    model.load_state_dict(torch.load(model_path))
    model.eval()

    img_ids = glob(os.path.join('inputs', args['dataset'], 'images', '*' + config['img_ext']))
    img_ids = [os.path.splitext(os.path.basename(p))[0] for p in img_ids]

    val_transform = Compose([
        transforms.Resize(config['input_h'], config['input_w']),
        transforms.Normalize(),
    ])

    val_dataset = Dataset(
        img_ids=img_ids,
        img_dir=os.path.join('inputs', args['dataset'], 'images'),
        mask_dir=os.path.join('inputs', args['dataset'], 'masks'),
        img_ext=config['img_ext'],
        mask_ext=config['mask_ext'],
        num_classes=config['num_classes'],
        transform=val_transform)

    ious, dices = [], []
    smooth = 1e-5
    with torch.no_grad():
        for i in tqdm(range(len(val_dataset))):
            img, gt, meta = val_dataset[i]

            img_tensor = torch.from_numpy(img).unsqueeze(0).cuda()
            gt_tensor = torch.from_numpy(gt).unsqueeze(0).cuda()

            # Predict original
            if config['deep_supervision']:
                p0 = torch.sigmoid(model(img_tensor)[-1])
            else:
                p0 = torch.sigmoid(model(img_tensor))

            if args['tta_mode'] == 'none':
                p_mean = p0
            elif args['tta_mode'] == 'horizontal':
                # Horizontal flip TTA
                img_h = torch.flip(img_tensor, dims=[3])
                if config['deep_supervision']:
                    p_h = torch.sigmoid(model(img_h)[-1])
                else:
                    p_h = torch.sigmoid(model(img_h))
                p_h = torch.flip(p_h, dims=[3])
                p_mean = (p0 + p_h) / 2.0
            elif args['tta_mode'] == 'all':
                # 4-flips TTA
                img_h = torch.flip(img_tensor, dims=[3])
                if config['deep_supervision']:
                    p_h = torch.sigmoid(model(img_h)[-1])
                else:
                    p_h = torch.sigmoid(model(img_h))
                p_h = torch.flip(p_h, dims=[3])

                img_v = torch.flip(img_tensor, dims=[2])
                if config['deep_supervision']:
                    p_v = torch.sigmoid(model(img_v)[-1])
                else:
                    p_v = torch.sigmoid(model(img_v))
                p_v = torch.flip(p_v, dims=[2])

                img_hv = torch.flip(img_tensor, dims=[2, 3])
                if config['deep_supervision']:
                    p_hv = torch.sigmoid(model(img_hv)[-1])
                else:
                    p_hv = torch.sigmoid(model(img_hv))
                p_hv = torch.flip(p_hv, dims=[2, 3])

                p_mean = (p0 + p_h + p_v + p_hv) / 4.0

            # Calculate IoU (hard threshold at 0.5)
            pred_bin = p_mean > 0.5
            gt_bin = gt_tensor > 0.5
            intersection = (pred_bin & gt_bin).sum()
            union = (pred_bin | gt_bin).sum()
            iou = (intersection.float() + smooth) / (union.float() + smooth)
            ious.append(iou.item())

            # Calculate DICE (soft Dice score, exactly matching metrics.py)
            intersection_soft = (p_mean * gt_tensor).sum()
            dice = (2. * intersection_soft + smooth) / (p_mean.sum() + gt_tensor.sum() + smooth)
            dices.append(dice.item())

    print('TTA Mode: %s' % args['tta_mode'])
    print('TTA IoU: %.4f' % np.mean(ious))
    print('TTA Dice: %.4f' % np.mean(dices))

if __name__ == "__main__":
    main()
"""

with open('eval_tta.py', 'w') as f:
    f.write(eval_tta_code)
print("eval_tta.py written.")

# 1. Evaluate baseline (no TTA) to verify correctness
print("\n=== BASELINE MODEL EVALUATION (No TTA) ===")
!python eval_tta.py --name busi-gs-25_NestedUNet_256_woDS --dataset busi-test-130 --tta_mode none

# 2. Evaluate baseline (Horizontal TTA)
print("\n=== BASELINE MODEL EVALUATION (Horizontal TTA) ===")
!python eval_tta.py --name busi-gs-25_NestedUNet_256_woDS --dataset busi-test-130 --tta_mode horizontal

# 3. Evaluate baseline (4-Flip TTA)
print("\n=== BASELINE MODEL EVALUATION (4-Flip TTA) ===")
!python eval_tta.py --name busi-gs-25_NestedUNet_256_woDS --dataset busi-test-130 --tta_mode all

# 4. Evaluate improved model (no TTA) to verify correctness
print("\n=== IMPROVED MODEL EVALUATION (No TTA) ===")
!python eval_tta.py --name busi-aug-train_NestedUNet_256_woDS --dataset busi-test-130 --tta_mode none

# 5. Evaluate improved model (Horizontal TTA)
print("\n=== IMPROVED MODEL EVALUATION (Horizontal TTA) ===")
!python eval_tta.py --name busi-aug-train_NestedUNet_256_woDS --dataset busi-test-130 --tta_mode horizontal

# 6. Evaluate improved model (4-Flip TTA)
print("\n=== IMPROVED MODEL EVALUATION (4-Flip TTA) ===")
!python eval_tta.py --name busi-aug-train_NestedUNet_256_woDS --dataset busi-test-130 --tta_mode all

%cd /content

/content/Auto-Generate-WLs/pytorch-nested-unet
eval_tta.py written.

=== BASELINE MODEL EVALUATION (No TTA) ===
100% 129/129 [00:06<00:00, 21.25it/s]
TTA Mode: none
TTA IoU: 0.4310
TTA Dice: 0.3846

=== BASELINE MODEL EVALUATION (Horizontal TTA) ===
100% 129/129 [00:08<00:00, 14.77it/s]
TTA Mode: horizontal
TTA IoU: 0.4394
TTA Dice: 0.3893

=== BASELINE MODEL EVALUATION (4-Flip TTA) ===
100% 129/129 [00:13<00:00,  9.49it/s]
TTA Mode: all
TTA IoU: 0.4409
TTA Dice: 0.3904

=== IMPROVED MODEL EVALUATION (No TTA) ===
100% 129/129 [00:05<00:00, 22.50it/s]
TTA Mode: none
TTA IoU: 0.3719
TTA Dice: 0.3966

=== IMPROVED MODEL EVALUATION (Horizontal TTA) ===
100% 129/129 [00:08<00:00, 15.94it/s]
TTA Mode: horizontal
TTA IoU: 0.3726
TTA Dice: 0.3986

=== IMPROVED MODEL EVALUATION (4-Flip TTA) ===
100% 129/129 [00:13<00:00,  9.67it/s]
TTA Mode: all
TTA IoU: 0.3836
TTA Dice: 0.4157
/content


In [ ]:
# ==============================================================================
# ## Section 19: Training directly on Paper's 25 GS + 100 WL Dataset
# ==============================================================================
%cd /content/Auto-Generate-WLs/pytorch-nested-unet

# Link the paper's pre-built 100 WL dataset from your Google Drive
!ln -sf /content/drive/MyDrive/inputs/busi-25-aug-100 inputs/busi-25-aug-100

# Train the model on the full 125 samples (25 GS + 100 WL)
!python train.py     --dataset busi-25-aug-100     --arch NestedUNet     --img_ext .png --mask_ext .png     --batch_size 4     --input_w 256 --input_h 256     --seed 42     --optimizer Adam --lr 3e-4     --epochs 200

# Evaluate on the unseen test set
print("\n=== EVALUATING 100 WL MODEL ON TEST SET ===")
!python eval.py     --name busi-25-aug-100_NestedUNet_256_woDS     --dataset busi-test-130

# Evaluate 100 WL Model with Horizontal TTA
print("\n=== EVALUATING 100 WL MODEL WITH HORIZONTAL TTA ===")
!python eval_tta.py     --name busi-25-aug-100_NestedUNet_256_woDS     --dataset busi-test-130 --tta_mode horizontal

%cd /content

/content/Auto-Generate-WLs/pytorch-nested-unet
--------------------
name: busi-25-aug-100_NestedUNet_256_woDS
epochs: 200
batch_size: 4
arch: NestedUNet
deep_supervision: False
input_channels: 3
num_classes: 1
input_w: 256
input_h: 256
loss: BCEDiceLoss
dataset: busi-25-aug-100
img_ext: .png
mask_ext: .png
optimizer: Adam
lr: 0.0003
momentum: 0.9
weight_decay: 0.0001
nesterov: False
scheduler: CosineAnnealingLR
min_lr: 1e-05
factor: 0.1
patience: 2
milestones: 1,2
gamma: 0.6666666666666666
early_stopping: -1
num_workers: 4
seed: 42
val_dataset: None
--------------------
=> creating model NestedUNet
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to av

In [ ]:
# ── Section 20: Train on Paper's 25 GS + 100 POINTS WL Dataset ────────────────
%cd /content/Auto-Generate-WLs/pytorch-nested-unet

# Link the points-prompt 100 WL dataset
!ln -sf /content/drive/MyDrive/inputs/busi-25-points-aug-100 inputs/busi-25-points-aug-100

# Train on points-prompt dataset
!python train.py     --dataset busi-25-points-aug-100     --arch NestedUNet     --img_ext .png --mask_ext .png     --batch_size 4     --input_w 256 --input_h 256     --seed 42     --optimizer Adam --lr 3e-4     --epochs 200

# Evaluate
print("\n=== EVALUATING POINTS 100 WL MODEL ON TEST SET ===")
!python eval.py     --name busi-25-points-aug-100_NestedUNet_256_woDS     --dataset busi-test-130

# TTA
print("\n=== EVALUATING POINTS 100 WL MODEL WITH NO TTA ===")
!python eval_tta.py     --name busi-25-points-aug-100_NestedUNet_256_woDS     --dataset busi-test-130 --tta_mode none

print("\n=== EVALUATING POINTS 100 WL MODEL WITH HORIZONTAL TTA ===")
!python eval_tta.py     --name busi-25-points-aug-100_NestedUNet_256_woDS     --dataset busi-test-130 --tta_mode horizontal

%cd /content

/content/Auto-Generate-WLs/pytorch-nested-unet
--------------------
name: busi-25-points-aug-100_NestedUNet_256_woDS
epochs: 200
batch_size: 4
arch: NestedUNet
deep_supervision: False
input_channels: 3
num_classes: 1
input_w: 256
input_h: 256
loss: BCEDiceLoss
dataset: busi-25-points-aug-100
img_ext: .png
mask_ext: .png
optimizer: Adam
lr: 0.0003
momentum: 0.9
weight_decay: 0.0001
nesterov: False
scheduler: CosineAnnealingLR
min_lr: 1e-05
factor: 0.1
patience: 2
milestones: 1,2
gamma: 0.6666666666666666
early_stopping: -1
num_workers: 4
seed: 42
val_dataset: None
--------------------
=> creating model NestedUNet
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worke

In [ ]:
!cp -r /content/Auto-Generate-WLs/pytorch-nested-unet/models/busi-aug-train_NestedUNet_256_woDS /content/drive/MyDrive/models/

In [ ]:
# Save Section 19 model (Box-prompt 100 WL)
!cp -r /content/Auto-Generate-WLs/pytorch-nested-unet/models/busi-25-aug-100_NestedUNet_256_woDS /content/drive/MyDrive/models/

In [ ]:

# Save Section 20 model (Points-prompt 100 WL)
!cp -r /content/Auto-Generate-WLs/pytorch-nested-unet/models/busi-25-points-aug-100_NestedUNet_256_woDS /content/drive/MyDrive/models/